# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [2]:
from pathlib import Path
import gc
import math
import shutil

import duckdb
import polars as pl

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)

# --- Path -------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():      # jika notebook dijalankan dari folder notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ID_REG_KG_2511'
DATA_PATH = str(DATA_DIR / '*.parquet')

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih'            # hasil utama (per file)
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'   # opsional (1 file)

valid_files = [f for f in sorted(DATA_DIR.glob('*.parquet')) if f.stat().st_size > 0]
if not valid_files:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di {DATA_DIR}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

# --- LazyFrame: hanya membaca metadata, data TIDAK dimuat ke RAM ---------
df = pl.scan_parquet(DATA_PATH)
SCHEMA = df.collect_schema()
SEMUA_KOLOM = SCHEMA.names()

# --- Pengelompokan kolom (dipakai di seluruh notebook) -----------------
KOLOM_VEKTOR = [c for c in ('embedding', 'tfidf_vector') if c in SEMUA_KOLOM]
KOLOM_JSON = [c for c in SEMUA_KOLOM if c.endswith('_json')]
KOLOM_TEKS_PANJANG = [c for c in ('about', 'content') if c in SEMUA_KOLOM]
KOLOM_KATEGORIK = [c for c in (
    'regulation_type', 'enacting_body', 'regulation_number', 'year',
    'effective_date', 'chapter', 'article', 'kg_primary_domain') if c in SEMUA_KOLOM]
KOLOM_SKOR = [c for c in (
    'kg_domain_confidence', 'kg_authority_score', 'kg_temporal_score',
    'kg_legal_richness', 'kg_legal_complexity', 'kg_completeness_score',
    'kg_connectivity_score') if c in SEMUA_KOLOM]               # seharusnya 0..1
KOLOM_HITUNG = [c for c in ('kg_entity_count', 'kg_cross_ref_count') if c in SEMUA_KOLOM]

# Nilai yang sebenarnya berarti "kosong" tetapi bukan NULL
PLACEHOLDER = ['', 'unknown', 'n/a', 'null', 'none', '-', 'tidak diketahui']

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('Jumlah file  :', len(valid_files))
print('Jumlah kolom :', len(SEMUA_KOLOM))
print('Vektor       :', KOLOM_VEKTOR)
print('JSON         :', len(KOLOM_JSON), 'kolom')
print('Skor (0..1)  :', KOLOM_SKOR)
print('Kolom hitung :', KOLOM_HITUNG)


def jalankan_per_file(ekspresi, transform=None):
    """Hitung `ekspresi` (agregat 1 baris) pada tiap file satu per satu, lalu jumlahkan.
    `transform` (opsional) = fungsi cleaning yang diterapkan ke tiap file sebelum dihitung.
    Dengan cara ini RAM hanya terpakai sebesar 1 file."""
    hasil = []
    for file in valid_files:
        lf = pl.scan_parquet(file)
        if transform is not None:
            lf = transform(lf)
        hasil.append(lf.select(ekspresi).collect())
    gc.collect()
    return pl.concat(hasil).sum()

PROJECT_ROOT : /workspaces/tugas1-Alfian79-2026
DATA_DIR     : /workspaces/tugas1-Alfian79-2026/data/raw/ID_REG_KG_2511
Jumlah file  : 1
Jumlah kolom : 41
Vektor       : ['embedding', 'tfidf_vector']
JSON         : 9 kolom
Skor (0..1)  : ['kg_domain_confidence', 'kg_authority_score', 'kg_temporal_score', 'kg_legal_richness', 'kg_legal_complexity', 'kg_completeness_score', 'kg_connectivity_score']
Kolom hitung : ['kg_entity_count', 'kg_cross_ref_count']


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [7]:
con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")
con.execute("SET temp_directory = '/tmp/duckdb_temp'")

kolom_ringan = [c for c in SEMUA_KOLOM if c not in KOLOM_VEKTOR + KOLOM_JSON + KOLOM_TEKS_PANJANG]
select_ringan = ', '.join(f'"{c}"' for c in kolom_ringan)
glob_sql = DATA_PATH.replace("'", "''")

parquet_files = [str(p) for p in DATA_DIR.glob('*.parquet') if p.stat().st_size > 0]

if not parquet_files:
    raise FileNotFoundError(f"Tidak ada file Parquet valid di {DATA_DIR}")

# Buat string list path untuk query DuckDB dengan aman
read_list = ", ".join(f"'{p}'" for p in parquet_files)

profil = con.execute(
    f"""
    SUMMARIZE
    SELECT {select_ringan},
           length(CAST(about AS VARCHAR)) AS about_len,
           length(CAST(content AS VARCHAR)) AS content_len
    FROM read_parquet([{read_list}])
    """
).pl()

# Tampilkan versi ringkas (nilai min/max teks dipotong supaya tabel terbaca)
profil_ringkas = profil.select(
    'column_name', 'column_type',
    pl.col('min').str.slice(0, 25),
    pl.col('max').str.slice(0, 25),
    'approx_unique', 'avg', 'std', 'q50', 'count',
    pl.col('null_percentage').cast(pl.Float64),
)
profil_ringkas

column_name,column_type,min,max,approx_unique,avg,std,q50,count,null_percentage
str,str,str,str,i64,str,str,str,i64,f64
"""global_id""","""BIGINT""","""0""","""9999""",8571,"""4999.5""","""2886.8956799071675""","""5000""",10000,0.0
"""local_id""","""BIGINT""","""0""","""9999""",8571,"""4999.5""","""2886.8956799071675""","""5000""",10000,0.0
"""regulation_type""","""VARCHAR""","""PERATURAN BUPATI""","""Unknown""",12,null,null,null,10000,0.0
"""enacting_body""","""VARCHAR""","""AGAM""","""WALIKOTA TANGERANG SELATA""",81,null,null,null,10000,0.0
"""regulation_number""","""VARCHAR""","""-""","""URUT""",134,null,null,null,10000,0.0
"""year""","""VARCHAR""","""1945""","""2017""",43,null,null,null,10000,0.0
"""effective_date""","""VARCHAR""","""02 Januari 2014""","""Unknown""",225,null,null,null,10000,0.0
"""chapter""","""VARCHAR""","""BAB I""","""PENUTUP""",927,null,null,null,10000,0.0
"""article""","""VARCHAR""","""LAMPIRAN""","""Pengesahan""",459,null,null,null,10000,0.0


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [8]:
KOLOM_PERIKSA_PLACEHOLDER = KOLOM_KATEGORIK + KOLOM_TEKS_PANJANG


def adalah_placeholder(kolom):
    return pl.col(kolom).str.strip_chars().str.to_lowercase().is_in(PLACEHOLDER)


ekspresi = [pl.len().alias('__baris__')]
ekspresi += [pl.col(c).null_count().alias(f'null__{c}') for c in SEMUA_KOLOM]
ekspresi += [adalah_placeholder(c).sum().alias(f'ph__{c}') for c in KOLOM_PERIKSA_PLACEHOLDER]
ekspresi += [(pl.col(c).list.len() == 0).sum().alias(f'kosong__{c}') for c in KOLOM_VEKTOR]

hasil = jalankan_per_file(ekspresi).row(0, named=True)
jumlah_baris = int(hasil['__baris__'])
print('Jumlah seluruh baris:', f'{jumlah_baris:,}')

skema_ringkasan = {'kolom': pl.String, 'jumlah': pl.Int64, 'persen': pl.Float64}


def buat_ringkasan(prefiks, daftar_kolom, nama_jumlah, nama_persen):
    baris = [
        {'kolom': c, 'jumlah': int(hasil[f'{prefiks}{c}']),
         'persen': round(int(hasil[f'{prefiks}{c}']) / jumlah_baris * 100, 2)}
        for c in daftar_kolom if int(hasil[f'{prefiks}{c}']) > 0
    ]
    return (pl.DataFrame(baris, schema=skema_ringkasan)
            .rename({'jumlah': nama_jumlah, 'persen': nama_persen})
            .sort(nama_jumlah, descending=True))


# (1) NULL asli
null_summary = buat_ringkasan('null__', SEMUA_KOLOM, 'jumlah_null', 'persen_null')
print('Kolom dengan NULL asli:', null_summary.height)
display(null_summary)

# (2) Missing tersembunyi ('Unknown', '', spasi, dst.)
placeholder_summary = buat_ringkasan('ph__', KOLOM_PERIKSA_PLACEHOLDER, 'jumlah_placeholder', 'persen_placeholder')
print('Kolom dengan missing tersembunyi:', placeholder_summary.height)
display(placeholder_summary)

# (3) List kosong pada kolom vektor
for c in KOLOM_VEKTOR:
    print(f'{c}: list kosong = {int(hasil[f"kosong__{c}"]):,} | NULL = {int(hasil[f"null__{c}"]):,}')

# Isi kolom yang bermasalah (untuk melihat bentuk missing tersembunyinya)
if 'regulation_type' in SEMUA_KOLOM:
    display(df.group_by('regulation_type').len().sort('len', descending=True).collect())

Jumlah seluruh baris: 10,000
Kolom dengan NULL asli: 0


kolom,jumlah_null,persen_null
str,i64,f64


Kolom dengan missing tersembunyi: 5


kolom,jumlah_placeholder,persen_placeholder
str,i64,f64
"""effective_date""",2810,28.1
"""chapter""",1788,17.88
"""regulation_type""",806,8.06
"""enacting_body""",460,4.6
"""regulation_number""",207,2.07


embedding: list kosong = 0 | NULL = 0
tfidf_vector: list kosong = 0 | NULL = 0


regulation_type,len
str,u32
"""PERATURAN PEMERINTAH""",5636
"""PERATURAN DAERAH KABUPATEN""",1379
"""Unknown""",806
"""PERATURAN BUPATI""",781
"""UNDANG-UNDANG""",678
"""PERATURAN WALIKOTA""",284
"""PERATURAN DAERAH KOTA""",268
"""PERATURAN GUBERNUR""",106
"""PERATURAN DAERAH""",28


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [9]:
KOLOM_PLACEHOLDER = placeholder_summary['kolom'].to_list()

# Median untuk kolom numerik yang punya NULL asli (kosong jika tidak ada)
KOLOM_NULL_NUMERIK = [c for c in null_summary['kolom'].to_list()
                      if c not in KOLOM_VEKTOR + KOLOM_JSON and SCHEMA[c].is_numeric()]
MEDIAN_NUMERIK = {c: df.select(pl.col(c).median()).collect().item() for c in KOLOM_NULL_NUMERIK}


def tangani_missing(lf):
    ekspresi = [
        pl.when(adalah_placeholder(c)).then(None).otherwise(pl.col(c)).alias(c)
        for c in KOLOM_PLACEHOLDER
    ]
    ekspresi += [pl.col(c).fill_null(m) for c, m in MEDIAN_NUMERIK.items()]
    return lf.with_columns(ekspresi) if ekspresi else lf


df_missing_handled = tangani_missing(df)

# Verifikasi: hitung NULL pada kolom yang ditangani, sebelum vs sesudah
if KOLOM_PLACEHOLDER:
    sesudah = jalankan_per_file(
        [pl.col(c).null_count().alias(c) for c in KOLOM_PLACEHOLDER],
        transform=tangani_missing,
    ).row(0, named=True)
    perbandingan = pl.DataFrame([
        {'kolom': c,
         'null_sebelum': int(hasil[f'null__{c}']),
         'placeholder_diubah': int(hasil[f'ph__{c}']),
         'null_sesudah': int(sesudah[c])}
        for c in KOLOM_PLACEHOLDER
    ])
    display(perbandingan)
else:
    print('Tidak ada missing tersembunyi yang perlu diubah.')

print('NULL numerik yang diisi median:', MEDIAN_NUMERIK or 'tidak ada')

kolom,null_sebelum,placeholder_diubah,null_sesudah
str,i64,i64,i64
"""effective_date""",0,2810,2810
"""chapter""",0,1788,1788
"""regulation_type""",0,806,806
"""enacting_body""",0,460,460
"""regulation_number""",0,207,207


NULL numerik yang diisi median: tidak ada


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [10]:
# --- A. Kolom hitungan: batas IQR (dihitung dari seluruh data, 2 kolom Int64 saja = ringan) ---
kuantil = df_missing_handled.select(
    [pl.col(c).quantile(0.25).alias(f'{c}__q1') for c in KOLOM_HITUNG]
    + [pl.col(c).quantile(0.75).alias(f'{c}__q3') for c in KOLOM_HITUNG]
    + [pl.col(c).quantile(0.99).alias(f'{c}__p99') for c in KOLOM_HITUNG]
    + [pl.col(c).max().alias(f'{c}__max') for c in KOLOM_HITUNG]
).collect().row(0, named=True)

BATAS_OUTLIER = {}      # {kolom: (batas_bawah, batas_atas)} sebagai bilangan bulat
for c in KOLOM_HITUNG:
    q1, q3 = kuantil[f'{c}__q1'], kuantil[f'{c}__q3']
    iqr = q3 - q1
    bawah = max(0, math.ceil(q1 - 1.5 * iqr))                   # hitungan tidak boleh negatif
    atas = math.floor(q3 + 1.5 * iqr) if iqr > 0 else int(kuantil[f'{c}__p99'])
    BATAS_OUTLIER[c] = (bawah, atas)
    print(f'{c}: Q1={q1}, Q3={q3}, IQR={iqr} -> batas bawah={bawah}, batas atas={atas}')

jumlah_outlier = df_missing_handled.select(
    [((pl.col(c) < b) | (pl.col(c) > a)).sum().alias(c) for c, (b, a) in BATAS_OUTLIER.items()]
).collect().row(0, named=True)

tabel_outlier = pl.DataFrame([
    {'kolom': c,
     'batas_bawah': b, 'batas_atas': a,
     'nilai_max': int(kuantil[f'{c}__max']),
     'jumlah_outlier': int(jumlah_outlier[c]),
     'persen_outlier': round(int(jumlah_outlier[c]) / jumlah_baris * 100, 2)}
    for c, (b, a) in BATAS_OUTLIER.items()
])
display(tabel_outlier)

# --- B. Kolom skor: periksa pelanggaran rentang 0..1 -------------------------------------
pelanggaran_skor = df_missing_handled.select(
    [((pl.col(c) < 0) | (pl.col(c) > 1)).sum().alias(c) for c in KOLOM_SKOR]
).collect().row(0, named=True)
nilai_max_skor = df_missing_handled.select([pl.col(c).max().alias(c) for c in KOLOM_SKOR]).collect().row(0, named=True)

tabel_skor = pl.DataFrame([
    {'kolom': c, 'nilai_max': nilai_max_skor[c], 'di_luar_0_1': int(pelanggaran_skor[c])}
    for c in KOLOM_SKOR
])
display(tabel_skor)


# --- C. Penanganan: windsorization (clip), bukan menghapus baris ---------------------------
def tangani_outlier(lf):
    penanda = [((pl.col(c) < b) | (pl.col(c) > a)).alias(f'{c}_outlier')       # dihitung SEBELUM clip
               for c, (b, a) in BATAS_OUTLIER.items()]
    clip_hitungan = [pl.col(c).clip(lower_bound=b, upper_bound=a)
                     for c, (b, a) in BATAS_OUTLIER.items()]
    clip_skor = [pl.col(c).clip(lower_bound=0.0, upper_bound=1.0) for c in KOLOM_SKOR]
    return lf.with_columns(penanda).with_columns(clip_hitungan + clip_skor)


df_outlier_handled = tangani_outlier(df_missing_handled)

kg_entity_count: Q1=7.0, Q3=20.0, IQR=13.0 -> batas bawah=0, batas atas=39
kg_cross_ref_count: Q1=0.0, Q3=3.0, IQR=3.0 -> batas bawah=0, batas atas=7


kolom,batas_bawah,batas_atas,nilai_max,jumlah_outlier,persen_outlier
str,i64,i64,i64,i64,f64
"""kg_entity_count""",0,39,561,995,9.95
"""kg_cross_ref_count""",0,7,267,1002,10.02


kolom,nilai_max,di_luar_0_1
str,f64,i64
"""kg_domain_confidence""",1.0,0
"""kg_authority_score""",0.86,0
"""kg_temporal_score""",0.7,0
"""kg_legal_richness""",1.0,0
"""kg_legal_complexity""",0.815,0
"""kg_completeness_score""",1.0,0
"""kg_connectivity_score""",1.139375,34


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [11]:
KUNCI_DEDUP = [c for c in ('regulation_type', 'regulation_number', 'year',
                           'article', 'chunk_id', 'content') if c in SEMUA_KOLOM]


def ekspresi_hash(daftar_kolom):
    return pl.concat_str(
        [pl.col(c).cast(pl.String).fill_null('<NULL>') for c in daftar_kolom],
        separator='\x1f',
    ).hash(seed=0)


# Bangun tabel hash per file (RAM hanya sebesar 1 file; hasil akhirnya kecil)
potongan = []
for file in valid_files:
    potongan.append(
        tangani_missing(pl.scan_parquet(file)).select(
            'global_id',
            ekspresi_hash(KUNCI_DEDUP).alias('hash_kunci'),
            ekspresi_hash(['content']).alias('hash_content'),
        ).collect()
    )
tabel_hash = pl.concat(potongan)
del potongan
gc.collect()

total_baris = tabel_hash.height
unik_global_id = tabel_hash['global_id'].n_unique()
unik_kunci = tabel_hash['hash_kunci'].n_unique()
unik_content = tabel_hash['hash_content'].n_unique()

print(f'Total baris                              : {total_baris:,}')
print(f'Baris unik berdasarkan global_id         : {unik_global_id:,}  (ID surrogate, selalu unik)')
print(f'Baris unik berdasarkan kunci bisnis      : {unik_kunci:,}')
print(f'Duplikat berdasarkan kunci bisnis        : {total_baris - unik_kunci:,}')
print(f'(info) duplikat jika hanya content sama  : {total_baris - unik_content:,}')

# Pertahankan baris pertama (global_id terkecil) tiap kunci, buang sisanya
pertahankan = tabel_hash.sort('global_id').unique(subset='hash_kunci', keep='first', maintain_order=True)
ID_DIBUANG = (tabel_hash.join(pertahankan.select('global_id'), on='global_id', how='anti')
              .select('global_id'))
print(f'Baris yang akan dibuang                  : {ID_DIBUANG.height:,}')

if ID_DIBUANG.height > 0:   # tampilkan contoh baris duplikat yang dibuang
    display(
        df_missing_handled
        .join(ID_DIBUANG.head(5).lazy(), on='global_id', how='semi')
        .select('global_id', *[c for c in KUNCI_DEDUP if c != 'content'],
                pl.col('content').str.slice(0, 80).alias('content_awal'))
        .collect()
    )


def hapus_duplikat(lf):
    return lf.join(ID_DIBUANG.lazy(), on='global_id', how='anti')


def terapkan_cleaning(lf):
    """Seluruh aturan cleaning dalam satu fungsi (dipakai di validasi dan saat menyimpan)."""
    return hapus_duplikat(tangani_outlier(tangani_missing(lf)))


df_dedup = hapus_duplikat(df_outlier_handled)

Total baris                              : 10,000
Baris unik berdasarkan global_id         : 10,000  (ID surrogate, selalu unik)
Baris unik berdasarkan kunci bisnis      : 9,989
Duplikat berdasarkan kunci bisnis        : 11
(info) duplikat jika hanya content sama  : 881
Baris yang akan dibuang                  : 11


global_id,regulation_type,regulation_number,year,article,chunk_id,content_awal
i64,str,str,str,str,i64,str
1388,"""PERATURAN PEMERINTAH""","""70""","""2007""","""Pembukaan""",1,"""Menimbang : PERATURAN PEMERINTAH REPUBLIK INDONESIA NOMOR 70…"
1389,"""PERATURAN PEMERINTAH""","""70""","""2007""","""Pasal 1""",1,"""Negara Republik Indonesia melakukan penambahan penyertaan mo…"
1390,"""PERATURAN PEMERINTAH""","""70""","""2007""","""Pasal 2""",1,"""- (1) Penambahan penyertaan modal negara sebagaimana dimaksu…"
1391,"""PERATURAN PEMERINTAH""","""70""","""2007""","""Pasal 3""",1,"""Peraturan Pemerintah ini mulai berlaku pada tanggal diundang…"
1392,"""PERATURAN PEMERINTAH""","""70""","""2007""","""Pengesahan""",1,"""Penetapan: Agar setiap orang mengetahuinya, memerintahkan pe…"


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [12]:
# Tahun acuan untuk kg_years_old: ditentukan dari data (nilai terbanyak dari year + kg_years_old)
tahun_acuan = (
    df.select((pl.col('year').cast(pl.Int64, strict=False) + pl.col('kg_years_old')).alias('tahun_acuan'))
    .drop_nulls().group_by('tahun_acuan').len().sort('len', descending=True).head(3).collect()
)
display(tahun_acuan)
REF_TAHUN = int(tahun_acuan['tahun_acuan'][0])
print('Tahun acuan kg_years_old:', REF_TAHUN)


def ekspresi_validasi():
    tahun = pl.col('year').cast(pl.Int64, strict=False)
    return [
        pl.len().alias('total_baris'),
        pl.sum_horizontal(
            [adalah_placeholder(c).fill_null(False).cast(pl.Int64) for c in KOLOM_PERIKSA_PLACEHOLDER]
        ).sum().alias('teks_placeholder'),
        pl.sum_horizontal(
            [((pl.col(c) < 0) | (pl.col(c) > 1)).fill_null(False).cast(pl.Int64) for c in KOLOM_SKOR]
        ).sum().alias('skor_di_luar_0_1'),
        (~tahun.is_between(1945, 2026)).fill_null(True).sum().alias('year_tidak_valid'),
        ((tahun + pl.col('kg_years_old')) != REF_TAHUN).fill_null(False).sum().alias('years_old_tidak_konsisten'),
        (pl.col('content').str.strip_chars().str.len_chars() == 0).fill_null(True).sum().alias('content_kosong_atau_null'),
        pl.sum_horizontal(
            [(pl.col(c) < 0).fill_null(False).cast(pl.Int64) for c in KOLOM_HITUNG]
        ).sum().alias('hitungan_negatif'),
        pl.sum_horizontal(
            [((pl.col(c) < b) | (pl.col(c) > a)).fill_null(False).cast(pl.Int64)
             for c, (b, a) in BATAS_OUTLIER.items()]
        ).sum().alias('hitungan_di_atas_batas_iqr'),
    ]


sebelum = jalankan_per_file(ekspresi_validasi()).row(0, named=True)
sesudah = jalankan_per_file(ekspresi_validasi(), transform=terapkan_cleaning).row(0, named=True)

# Aturan duplikat dihitung dari tabel hash (lintas file)
tabel_sisa = tabel_hash.join(ID_DIBUANG, on='global_id', how='anti')
sebelum['duplikat_kunci_bisnis'] = total_baris - unik_kunci
sesudah['duplikat_kunci_bisnis'] = tabel_sisa.height - tabel_sisa['hash_kunci'].n_unique()

validasi = pl.DataFrame([
    {'aturan': k,
     'sebelum': int(sebelum[k]),
     'sesudah': int(sesudah[k]),
     'status': '-' if k == 'total_baris' else ('OK' if int(sesudah[k]) == 0 else 'PERIKSA')}
    for k in sebelum
])
validasi

tahun_acuan,len
i64,u32
2024,10000


Tahun acuan kg_years_old: 2024


aturan,sebelum,sesudah,status
str,i64,i64,str
"""total_baris""",10000,9989,"""-"""
"""teks_placeholder""",6071,0,"""OK"""
"""skor_di_luar_0_1""",34,0,"""OK"""
"""year_tidak_valid""",0,0,"""OK"""
"""years_old_tidak_konsisten""",0,0,"""OK"""
"""content_kosong_atau_null""",0,0,"""OK"""
"""hitungan_negatif""",0,0,"""OK"""
"""hitungan_di_atas_batas_iqr""",1997,0,"""OK"""
"""duplikat_kunci_bisnis""",11,0,"""OK"""


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [13]:
import pyarrow as pa
import pyarrow.parquet as pq

# Bebaskan memori dari langkah sebelumnya (tidak dipakai lagi di langkah ini)
for nama in ('con', 'tabel_sisa', 'tabel_hash', 'pertahankan', 'sebelum', 'sesudah'):
    globals().pop(nama, None)
gc.collect()

ULANG_DARI_AWAL = False     # True = hapus hasil lama dan proses ulang semua file (pakai jika aturan cleaning berubah)
UKURAN_BATCH = 1000         # baris per batch; turunkan ke 1000 jika masih crash

if ULANG_DARI_AWAL and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for i, file in enumerate(valid_files, start=1):
    tujuan = OUTPUT_DIR / file.name
    if tujuan.exists():                      # file yang sudah selesai dilewati (bisa dilanjutkan setelah crash)
        continue
    sementara = tujuan.with_suffix('.tmp')   # tulis ke .tmp dulu, ganti nama setelah selesai

    penulis = None
    for batch in pq.ParquetFile(file).iter_batches(batch_size=UKURAN_BATCH):
        bersih = terapkan_cleaning(pl.from_arrow(batch).lazy()).collect()
        tabel = bersih.to_arrow()
        if penulis is None:
            penulis = pq.ParquetWriter(sementara, tabel.schema, compression='zstd')
        penulis.write_table(tabel)
        del batch, bersih, tabel
    if penulis is not None:
        penulis.close()
        sementara.rename(tujuan)
    gc.collect()

    if i % 5 == 0 or i == len(valid_files):
        print(f'Berhasil: {i}/{len(valid_files)}')

# Verifikasi hasil simpan
jumlah_baris_akhir = (pl.scan_parquet(str(OUTPUT_DIR / '*.parquet'))
                      .select(pl.len()).collect().item())
jumlah_dibuang = jumlah_baris - jumlah_baris_akhir
ukuran_gb = sum(f.stat().st_size for f in OUTPUT_DIR.glob('*.parquet')) / 1e9

print(f'Baris sebelum cleaning : {jumlah_baris:,}')
print(f'Baris sesudah cleaning : {jumlah_baris_akhir:,}')
print(f'Baris dibuang          : {jumlah_dibuang:,}')
print(f'Ukuran folder          : {ukuran_gb:.2f} GB')
print(f'Data bersih tersimpan di: {OUTPUT_DIR}')

assert jumlah_dibuang == ID_DIBUANG.height, 'Jumlah baris dibuang tidak sesuai dengan hasil deduplikasi'
print('Kolom baru:', [c for c in pl.scan_parquet(str(OUTPUT_DIR / '*.parquet')).collect_schema().names()
                      if c not in SEMUA_KOLOM])

Berhasil: 1/1
Baris sebelum cleaning : 10,000
Baris sesudah cleaning : 9,989
Baris dibuang          : 11
Ukuran folder          : 0.04 GB
Data bersih tersimpan di: /workspaces/tugas1-Alfian79-2026/data/processed/dataset_bersih
Kolom baru: ['kg_entity_count_outlier', 'kg_cross_ref_count_outlier']


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.